# Technical note: Model structure and scoping

This note will briefly discuss Myokit's model structure and the scoping rules (which variable can access which).

It is **not** a replacement of the official syntax document: https://myokit.readthedocs.io/en/stable/syntax/model.html



## The basics

- A model has components.
- Components have variables.
- Variables can have further variables.

Variables fall into two groups, depending on their parent:

- Variables whose parent is a component are referred to as "component variables".
- Variables whose parent is a variable are referred to as "nested variables".

### Scope: Access

- Component variables are accessible from anywhere in the model
- Nested variables can be accessed only by their parent, siblings, or any of their descendants

To illustrate the nested variable rule, we use the tree of variables below

In [2]:
import myokit
m = myokit.Model()
q = m.add_component('q')
q.add_variable('t', rhs=0, binding='time')
a = q.add_variable('a', rhs=0)
b = a.add_variable('b', rhs=0)
c = a.add_variable('c', rhs=0)
d = b.add_variable('d', rhs=0)
e = b.add_variable('e', rhs=0)
f = c.add_variable('f', rhs=0)
g = d.add_variable('g', rhs=0)
vs = (a, b, c, d, e, f, g)

```
      a
     / \
    b   c
   / \   \ 
  d   e   f
 /
g
```

_"Nested variables can be accessed by their parent, siblings, or any of their descendants"_

# To-do double check the wording.

# To-do double check if parsing gives the same result

# To-do double check if variable access works the same way

In [3]:
for v in vs:
    ok = []
    for w in vs:
        w.set_rhs(myokit.Name(v))
        try:
            m.validate()
        except myokit.IntegrityError:
            pass
        else:
            ok.append(w)
        w.set_rhs(0)
    print(f'{v.name()} can be accessed by ' + ', '.join(w.name() for w in ok))

a can be accessed by b, c, d, e, f, g
b can be accessed by a, c, d, e, f, g
c can be accessed by a, b, d, e, f, g
d can be accessed by b, e, g
e can be accessed by b, d, g
f can be accessed by c
g can be accessed by d


Phrased the other way round, a variable can access its children, or children of any of its ancestors.
```
      a
     / \
    b   c
   / \   \ 
  d   e   f
 /
g
```

In [ ]:
vs = (a, b, c, d, e, f, g)
for v in vs:
    can_access = []
    for w in vs:
        v.set_rhs(myokit.Name(w))
        try:
            m.validate()
        except myokit.IntegrityError:
            pass
        else:
            can_access.append(w)
        v.set_rhs(0)
    print(f'{v.name()} can access ' + ', '.join(w.name() for w in can_access))

- `a` can access `b, c`
- `b` can access `a, c, d, e`
- `c` can access `a, b, f`
- `d` can access `a, b, c, e, g`
- `e` can access `a, b, c, d`
- `f` can access `a, b, c`
- `g` can access `a, b, c, d, e`